# Assembly Report Pipeline — Overview & Requirements


This notebook automates an **in-silico plasmid assembly workflow** and produces cleaned GenBank files for downstream use.

## What this notebook does
1. **Discover parts** in the folders `Promoter_parts/`, `RBS_parts/`, `Gene_of_interest_parts/`, `Terminator_parts/`, `Backbone_parts/`.
2. **Generate assembly reports** (`*_report.zip`) using `generate_assembly_reports`.
3. **Organize reports** by extracting `.gb/.gbk` into `reports/Assembly` via `organize_assembly_reports`.
4. **Clean duplicate features** in each GenBank: remove near-duplicates whose `(start, end)` are within `±tolerance` using `remove_near_duplicate_features` (in-place overwrite).

## Inputs & Outputs
- **Inputs:** the five parts folders listed above (absolute or relative to `BASE_DIR`), containing DNA part files (e.g. `.gb`, `.gbk`, `.dna`).
- **Outputs:** extracted and cleaned GenBank files in **`reports/Assembly/`** (cleaning overwrites in place and returns a `pathlib.Path`).

## Configuration (edit these first)
```python
from pathlib import Path

# Base directory; by default the current working directory
BASE_DIR = Path.cwd()

# Part folders
folder_paths = [
    BASE_DIR / "Promoter_parts",
    BASE_DIR / "RBS_parts",
    BASE_DIR / "Gene_of_interest_parts",
    BASE_DIR / "Terminator_parts",
    BASE_DIR / "Backbone_parts",
]

# Destination for generated reports; "reports/Assembly" will be created underneath
report_folder = BASE_DIR

# Optional: filter assemblies by substring (case-insensitive); set to "" or None to include all
gene_filter = "btADC"

In [1]:
# 1) Imports & Setup
from pathlib import Path
import os
from typing import Iterable

# Project functions:
# - organize_assembly_reports: scans *_report.zip and extracts all .gb/.gbk into a target folder
# - generate_assembly_reports: (folder-as-assembly mode) builds one assembly per folder of parts
# - remove_near_duplicate_features: optional helper to clean overlapping/duplicate features in SeqRecords
from assembly_designer.plasmidio import (
    organize_assembly_reports,
    generate_assembly_reports,
    remove_near_duplicate_features,
)

# (Optional) quick sanity check of optional dependencies used elsewhere:
# - dnacauldron: required for simulating assemblies and writing reports
# - snapgene_reader: enables reading .dna (SnapGene) files
try:
    import dnacauldron  # noqa: F401  # imported only to verify availability
    from snapgene_reader import snapgene_file_to_dict  # noqa: F401  # ditto
    print("Optional deps OK: dnacauldron, snapgene-reader")
except Exception as err:
    # If this prints, you can still proceed with GenBank-only workflows,
    # but .dna reading and DNAcauldron simulation/reporting may be unavailable.
    print("⚠️ Optional deps check:", err)



Optional deps OK: dnacauldron, snapgene-reader


In [2]:
# Base directory for your project. All part folders are expected inside this dir.
# Tip: If your notebook is not located next to the part folders, replace Path.cwd()
# with an absolute path, e.g. Path(r"C:\Repo\assembly_designer_test").
BASE_DIR = Path.cwd()

# Paths to the category folders (keep the order consistent with category_order below).
# You DO NOT need to change your folder structure; we just reference the existing ones.
folder_paths = [
    BASE_DIR / "SP",
    BASE_DIR / "GOI",
    BASE_DIR / "Backbone",
]

# Target folder for all generated reports (ZIPs) and extracted GenBank files.
# You can point this to a subfolder if you prefer, e.g. BASE_DIR / "reports".
report_folder = BASE_DIR

# Optional: a string to filter which gene(s) to include (by filename).
# Note: The combinatorial function shown earlier does not use this directly.
# If you want this filter applied, either rename files or extend the function
# to accept a filter. For now, it’s just a placeholder variable.
# gene_filter = "ecPanD.dna"  # or "" / None to include all


In [3]:
# Define the category order exactly as you want parts laid out in each assembly.
# This order also controls how assembly names are constructed.
category_order = ["SP", "GOI", "Backbone"]

# Convert your existing list of folders to a mapping: Category -> Folder path.
# The order of 'folder_paths' must match 'category_order' above.
category_dirs = dict(zip(category_order, folder_paths))

In [4]:
# Filter out non-existent folders (robust)
existing_paths = [str(p) for p in folder_paths if Path(p).exists()]
if not existing_paths:
    raise FileNotFoundError("No valid parts folders found — please check the paths.")

In [5]:
# Run the combinatorial assemblies:
# - Builds the cartesian product of the categories in 'category_order'
# - Creates one DNAcauldron simulation per combination
# - Writes one *_report.zip for each assembly into <report_folder>/reports
reports = generate_assembly_reports(
    category_dirs=category_dirs,
    category_order=category_order,
    output_dir=report_folder / "reports",
)

# Extract all constructed GenBank files (.gb/.gbk) from every report ZIP into
# <report_folder>/reports/Assembly and return the list of extracted file paths.
gb_paths = organize_assembly_reports(report_folder / "reports", reports, delete_zip=False)

print("Number of constructed GenBank files:", len(gb_paths))
print("First few:", [p.name for p in gb_paths[:10]])


c:\Users\tim\miniconda3\envs\adesigner\Lib\site-packages\Bio\SeqIO\InsdcIO.py:780: BiopythonWarning: Increasing length of locus line to allow long name. This will result in fields that are not in usual positions.
  warnings.warn(
findfont: Font family 'Inconsolata' not found.
findfont: Font family 'Inconsolata' not found.
findfont: Font family 'Inconsolata' not found.
findfont: Font family 'Inconsolata' not found.
findfont: Font family 'Inconsolata' not found.
findfont: Font family 'Inconsolata' not found.
findfont: Font family 'Inconsolata' not found.
findfont: Font family 'Inconsolata' not found.
findfont: Font family 'Inconsolata' not found.
findfont: Font family 'Inconsolata' not found.
findfont: Font family 'Inconsolata' not found.
findfont: Font family 'Inconsolata' not found.
findfont: Font family 'Inconsolata' not found.
findfont: Font family 'Inconsolata' not found.
findfont: Font family 'Inconsolata' not found.
findfont: Font family 'Inconsolata' not found.
findfont: Font fam

Number of constructed GenBank files: 24
First few: ['pSM124-DVA-CD-SP_NprE_pSM123-DVA-DE-cutinase_no_BsaI_pSM274.gb', 'pSM124-DVA-CD-SP_NprE_pSM123-DVA-DE-cutinase_no_BsaI_pSM277.gb', 'pSM124-DVA-CD-SP_NprE_pSM188-DVA_DE-wt-PETase_pSM274.gb', 'pSM124-DVA-CD-SP_NprE_pSM188-DVA_DE-wt-PETase_pSM277.gb', 'pSM124-DVA-CD-SP_NprE_pSM189-DVA-DE-PHL7_pSM274.gb', 'pSM124-DVA-CD-SP_NprE_pSM189-DVA-DE-PHL7_pSM277.gb', 'pSM124-DVA-CD-SP_NprE_pSM190-DVA_DE-FAST-PETase_pSM274.gb', 'pSM124-DVA-CD-SP_NprE_pSM190-DVA_DE-FAST-PETase_pSM277.gb', 'pSM125-DVA-CD-SP002_pSM123-DVA-DE-cutinase_no_BsaI_pSM274.gb', 'pSM125-DVA-CD-SP002_pSM123-DVA-DE-cutinase_no_BsaI_pSM277.gb']


In [6]:
# delete near-duplicate features in all extracted GenBank files
# (optional, but recommended if your parts have overlapping annotations)
assembly_folder = Path("reports") / "Assembly"

for file_name in os.listdir(assembly_folder):
    if file_name.endswith(".gb") or file_name.endswith(".gbk"):
        file_path = os.path.join(assembly_folder, file_name)
        remove_near_duplicate_features(file_path, tolerance=3)  # Tolerance of ±3 bases

✅ Cleaned file saved: reports\Assembly\pSM124-DVA-CD-SP_NprE_pSM123-DVA-DE-cutinase_no_BsaI_pSM274.gb
✅ Cleaned file saved: reports\Assembly\pSM124-DVA-CD-SP_NprE_pSM123-DVA-DE-cutinase_no_BsaI_pSM277.gb
✅ Cleaned file saved: reports\Assembly\pSM124-DVA-CD-SP_NprE_pSM188-DVA_DE-wt-PETase_pSM274.gb
✅ Cleaned file saved: reports\Assembly\pSM124-DVA-CD-SP_NprE_pSM188-DVA_DE-wt-PETase_pSM277.gb
✅ Cleaned file saved: reports\Assembly\pSM124-DVA-CD-SP_NprE_pSM189-DVA-DE-PHL7_pSM274.gb
✅ Cleaned file saved: reports\Assembly\pSM124-DVA-CD-SP_NprE_pSM189-DVA-DE-PHL7_pSM277.gb
✅ Cleaned file saved: reports\Assembly\pSM124-DVA-CD-SP_NprE_pSM190-DVA_DE-FAST-PETase_pSM274.gb
✅ Cleaned file saved: reports\Assembly\pSM124-DVA-CD-SP_NprE_pSM190-DVA_DE-FAST-PETase_pSM277.gb
✅ Cleaned file saved: reports\Assembly\pSM125-DVA-CD-SP002_pSM123-DVA-DE-cutinase_no_BsaI_pSM274.gb
✅ Cleaned file saved: reports\Assembly\pSM125-DVA-CD-SP002_pSM123-DVA-DE-cutinase_no_BsaI_pSM277.gb
✅ Cleaned file saved: reports\As